In [1]:
# 05_sensitivity.ipynb

# %% [markdown]
# # Cross-Domain AI Governance Gap Analysis
# ## 05. Sensitivity Analysis for the GGI
#
# Test whether the two headline findings survive perturbations of the GGI's design choices:
#   (F1) insurance domain shows a larger governance gap than credit
#   (F2) the regulation ordering of GGI (which regulation looks strictest)
#
# Four perturbation studies:
#   A. Threshold scaling      — shift all tau by a multiplicative factor
#   B. Aggregation rule       — Euclidean vs mean vs max over component gaps
#   C. Explainability threshold— vary tau[s_EXPL] alone (the dominant axis)
#   D. Bootstrap              — resample the test set to get GGI confidence intervals
#
# Component scores are reused from notebook 04 where possible; the bootstrap re-derives
# fairness/performance from stored predictions.

# %%
# Imports and configuration
import os
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import shap

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")

RANDOM_STATE = 42
SHAP_SAMPLE = 1000
N_BOOTSTRAP = 200
os.makedirs(TAB_DIR, exist_ok=True)

# Baseline thresholds (identical to notebook 04)
BASE_REGULATIONS = {
    "KR_AI_Basic_Act": {"s_PERF": 0.30, "s_DI": 0.80, "s_SP": 0.90, "s_EXPL": 0.50},
    "EU_AI_Act":       {"s_PERF": 0.40, "s_DI": 0.85, "s_SP": 0.92, "s_EXPL": 0.60},
    "NAIC":            {"s_PERF": 0.40, "s_DI": 0.90, "s_SP": 0.95, "s_EXPL": 0.55},
}
DATASETS = [
    ("German Credit",        "german"),
    ("Credit Card Approval", "credit_card_approval"),
    ("Medical Cost",         "insurance"),
    ("Porto Seguro",         "porto_seguro"),
]
DOMAIN_OF = {}

# %%
# Helpers (mirrors notebook 04)
with open(os.path.join(PROCESSED_DIR, "preprocess_meta.json")) as f:
    META = json.load(f)

def load_split(key):
    train = pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_train.parquet"))
    test = pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_test.parquet"))
    return train, test

def get_Xy(df, key):
    meta = META[key]
    target_col = meta["target_col"]
    sensitive_cols = meta["sensitive_cols"]
    aux_cols = [c for c in ["target_60dpd", "target_30dpd"] if c in df.columns and c != target_col]
    drop = [target_col] + sensitive_cols + aux_cols
    X = df.drop(columns=[c for c in drop if c in df.columns], errors="ignore")
    y = df[target_col].astype(int)
    return X, y

def group_di_spd(y_pred, priv_mask, unpriv_mask):
    ppr_p = y_pred[priv_mask].mean() if priv_mask.sum() else np.nan
    ppr_u = y_pred[unpriv_mask].mean() if unpriv_mask.sum() else np.nan
    if not (np.isfinite(ppr_p) and np.isfinite(ppr_u)) or ppr_p == 0:
        return np.nan, np.nan
    return ppr_u / ppr_p, ppr_u - ppr_p

def worst_case_fairness(test_df, sensitive_cols, y_pred):
    dis, spds = [], []
    for col in sensitive_cols:
        s = test_df[col]
        if s.nunique(dropna=True) < 2:
            continue
        privileged = s.mode(dropna=True).iloc[0]
        priv_mask = (s == privileged).values
        unpriv_mask = (s != privileged).values & s.notna().values
        di, spd = group_di_spd(y_pred, priv_mask, unpriv_mask)
        if np.isfinite(di): dis.append(di)
        if np.isfinite(spd): spds.append(spd)
    worst_di = min(dis) if dis else np.nan
    worst_abs_spd = max(abs(x) for x in spds) if spds else np.nan
    s_DI = min(1.0, worst_di) if np.isfinite(worst_di) else np.nan
    s_SP = 1.0 - worst_abs_spd if np.isfinite(worst_abs_spd) else np.nan
    return s_DI, s_SP

def shap_explainability(model, X, sample=SHAP_SAMPLE, seed=RANDOM_STATE):
    if len(X) > sample:
        X = X.sample(sample, random_state=seed)
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X)
    if isinstance(sv, list):
        sv = sv[-1]
    mean_abs = np.abs(sv).mean(axis=0)
    total = mean_abs.sum()
    if total == 0 or len(mean_abs) <= 1:
        return 0.0
    p = mean_abs / total
    p = p[p > 0]
    entropy = -np.sum(p * np.log(p))
    return float(1.0 - entropy / np.log(len(mean_abs)))

def component_gap(score, tau):
    if not np.isfinite(score):
        return np.nan
    return max(0.0, tau - score) / tau

def aggregate(gaps, rule="euclid"):
    gaps = [g for g in gaps if np.isfinite(g)]
    if not gaps:
        return np.nan
    if rule == "euclid":
        return float(np.sqrt(np.sum(np.square(gaps))))
    if rule == "mean":
        return float(np.mean(gaps))
    if rule == "max":
        return float(np.max(gaps))
    raise ValueError(rule)

def compute_ggi(scores, regulations, rule="euclid"):
    rows = []
    for name, sc in scores.items():
        for reg, tau in regulations.items():
            gaps = [
                component_gap(sc["s_PERF"], tau["s_PERF"]),
                component_gap(sc["s_DI"], tau["s_DI"]),
                component_gap(sc["s_SP"], tau["s_SP"]),
                component_gap(sc["s_EXPL"], tau["s_EXPL"]),
            ]
            rows.append({"domain": sc["domain"], "dataset": name,
                         "regulation": reg, "GGI": aggregate(gaps, rule)})
    return pd.DataFrame(rows)

# %%
# Recompute base component scores once (and keep fitted models + test frames for bootstrap)
component_scores = {}
fitted = {}

for display_name, key in DATASETS:
    train, test = load_split(key)
    X_train, y_train = get_Xy(train, key)
    X_test, y_test = get_Xy(test, key)
    X_test = X_test[X_train.columns]

    n_pos = int(y_train.sum()); n_neg = int((y_train == 0).sum())
    spw = (n_neg / n_pos) if n_pos > 0 else 1.0
    model = XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, scale_pos_weight=spw,
        eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1,
    )
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    thr = np.quantile(proba, 1.0 - float(y_train.mean()))
    y_pred = (proba >= thr).astype(int)

    auc = roc_auc_score(y_test.values, proba)
    s_PERF = max(0.0, (auc - 0.5) / 0.5)
    s_DI, s_SP = worst_case_fairness(test, META[key]["sensitive_cols"], y_pred)
    s_EXPL = shap_explainability(model, X_test)

    domain = META[key].get("domain", "")
    DOMAIN_OF[display_name] = domain
    component_scores[display_name] = {
        "domain": domain, "s_PERF": s_PERF, "s_DI": s_DI, "s_SP": s_SP, "s_EXPL": s_EXPL,
    }
    fitted[display_name] = {
        "key": key, "test": test, "X_test": X_test, "y_test": y_test,
        "proba": proba, "threshold": thr, "s_EXPL": s_EXPL,
    }
    print(f"[{display_name:>22}] base scores captured")

# Helper: does insurance > credit hold on a given GGI table (mean per domain)?
def insurance_gt_credit(ggi_table):
    m = ggi_table.groupby("domain")["GGI"].mean()
    if "insurance" in m and "credit" in m:
        return bool(m["insurance"] > m["credit"]), float(m["insurance"] - m["credit"])
    return np.nan, np.nan

# %%
# --- Study A: threshold scaling ---
# Multiply every tau by a factor; check whether F1 (insurance > credit) still holds.
factors = [0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15]
studyA = []
for f in factors:
    scaled = {reg: {k: min(v * f, 0.999) for k, v in tau.items()}
              for reg, tau in BASE_REGULATIONS.items()}
    ggi = compute_ggi(component_scores, scaled)
    holds, gap = insurance_gt_credit(ggi)
    dmean = ggi.groupby("domain")["GGI"].mean()
    studyA.append({"factor": f, "credit_mean": round(dmean.get("credit", np.nan), 4),
                   "insurance_mean": round(dmean.get("insurance", np.nan), 4),
                   "insurance_gt_credit": holds, "gap": round(gap, 4)})
studyA_df = pd.DataFrame(studyA)
print("Study A — threshold scaling")
print(studyA_df.to_string(index=False))
studyA_df.to_csv(os.path.join(TAB_DIR, "05_sensitivity_threshold.csv"), index=False)

# %%
# --- Study B: aggregation rule ---
# Recompute GGI under euclid/mean/max; check F1 and the regulation ordering (F2).
studyB = []
for rule in ["euclid", "mean", "max"]:
    ggi = compute_ggi(component_scores, BASE_REGULATIONS, rule=rule)
    holds, gap = insurance_gt_credit(ggi)
    reg_order = (ggi.groupby("regulation")["GGI"].mean()
                 .reindex(["KR_AI_Basic_Act", "EU_AI_Act", "NAIC"]).round(4))
    studyB.append({"rule": rule, "insurance_gt_credit": holds, "gap": round(gap, 4),
                   "KR": reg_order["KR_AI_Basic_Act"], "EU": reg_order["EU_AI_Act"],
                   "NAIC": reg_order["NAIC"]})
studyB_df = pd.DataFrame(studyB)
print("Study B — aggregation rule")
print(studyB_df.to_string(index=False))
studyB_df.to_csv(os.path.join(TAB_DIR, "05_sensitivity_aggregation.csv"), index=False)

# %%
# --- Study C: explainability threshold sweep ---
# The explainability axis dominates GGI, so vary tau[s_EXPL] across all regulations
# (uniformly) and see how domain means and F1 respond.
expl_taus = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
studyC = []
for et in expl_taus:
    reg = {r: {**tau, "s_EXPL": et} for r, tau in BASE_REGULATIONS.items()}
    ggi = compute_ggi(component_scores, reg)
    dmean = ggi.groupby("domain")["GGI"].mean()
    holds, gap = insurance_gt_credit(ggi)
    studyC.append({"tau_EXPL": et,
                   "credit_mean": round(dmean.get("credit", np.nan), 4),
                   "insurance_mean": round(dmean.get("insurance", np.nan), 4),
                   "insurance_gt_credit": holds, "gap": round(gap, 4)})
studyC_df = pd.DataFrame(studyC)
print("Study C — explainability threshold sweep")
print(studyC_df.to_string(index=False))
studyC_df.to_csv(os.path.join(TAB_DIR, "05_sensitivity_explainability.csv"), index=False)

# %%
# --- Study D: bootstrap confidence intervals for GGI ---
# Resample each test set with replacement; recompute performance + worst-case fairness
# (explainability is held at its base value, as SHAP per replicate is costly).
# Report GGI mean and 95% CI per dataset under the KR (baseline) regulation.
def bootstrap_ggi(dataset_name, n_boot=N_BOOTSTRAP, seed=RANDOM_STATE):
    info = fitted[dataset_name]
    key = info["key"]
    test = info["test"].reset_index(drop=True)
    proba = info["proba"]
    thr = info["threshold"]
    s_EXPL = info["s_EXPL"]
    y_test = info["y_test"].reset_index(drop=True).values
    sens_cols = META[key]["sensitive_cols"]

    rng = np.random.default_rng(seed)
    n = len(test)
    ggis = {"KR_AI_Basic_Act": [], "EU_AI_Act": [], "NAIC": []}

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yb = y_test[idx]
        pb = proba[idx]
        yhat = (pb >= thr).astype(int)
        # bootstrap AUC (guard against single-class resample)
        if len(np.unique(yb)) < 2:
            continue
        auc = roc_auc_score(yb, pb)
        s_PERF = max(0.0, (auc - 0.5) / 0.5)
        s_DI, s_SP = worst_case_fairness(test.iloc[idx], sens_cols, yhat)
        sc = {dataset_name: {"domain": DOMAIN_OF[dataset_name],
                             "s_PERF": s_PERF, "s_DI": s_DI, "s_SP": s_SP, "s_EXPL": s_EXPL}}
        g = compute_ggi(sc, BASE_REGULATIONS)
        for reg in ggis:
            v = g.loc[g["regulation"] == reg, "GGI"].iloc[0]
            if np.isfinite(v):
                ggis[reg].append(v)
    return ggis

studyD = []
for display_name, _ in DATASETS:
    ci = bootstrap_ggi(display_name)
    for reg, vals in ci.items():
        arr = np.array(vals)
        studyD.append({
            "dataset": display_name, "domain": DOMAIN_OF[display_name], "regulation": reg,
            "GGI_mean": round(arr.mean(), 4),
            "CI_low": round(np.percentile(arr, 2.5), 4),
            "CI_high": round(np.percentile(arr, 97.5), 4),
            "n_valid": len(arr),
        })
studyD_df = pd.DataFrame(studyD)
print("Study D — bootstrap 95% CI")
print(studyD_df.to_string(index=False))
studyD_df.to_csv(os.path.join(TAB_DIR, "05_sensitivity_bootstrap.csv"), index=False)

# %%
# --- Overall verdict ---
print("=== Robustness summary ===")
print("F1 (insurance > credit) holds across threshold factors:",
      studyA_df["insurance_gt_credit"].all())
print("F1 holds across aggregation rules:",
      studyB_df["insurance_gt_credit"].all())
print("F1 holds across explainability thresholds:",
      studyC_df["insurance_gt_credit"].all())

[         German Credit] base scores captured
[  Credit Card Approval] base scores captured
[          Medical Cost] base scores captured
[          Porto Seguro] base scores captured
Study A — threshold scaling
 factor  credit_mean  insurance_mean  insurance_gt_credit    gap
   0.85       0.5775          1.2019                 True 0.6243
   0.90       0.6010          1.2245                 True 0.6235
   0.95       0.6220          1.2459                 True 0.6239
   1.00       0.6418          1.2661                 True 0.6243
   1.05       0.6620          1.2859                 True 0.6239
   1.10       0.6814          1.3043                 True 0.6230
   1.15       0.6985          1.3195                 True 0.6210
Study B — aggregation rule
  rule  insurance_gt_credit    gap     KR     EU   NAIC
euclid                 True 0.6243 0.9108 0.9842 0.9669
  mean                 True 0.3630 0.3263 0.3587 0.3595
   max                 True 0.2237 0.7285 0.7737 0.7560
Study C — explain